# LLM Reads Web Page and Answers Questions

In [21]:
%%time
%pip install -U requests beautifulsoup4 transformers torch ipython-autotime ipywidgets

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
CPU times: user 7.82 ms, sys: 22.9 ms, total: 30.7 ms
Wall time: 1.17 s
time: 1.17 s (started: 2025-04-16 09:08:36 -07:00)


In [22]:
%%time
import warnings
warnings.filterwarnings("ignore")

CPU times: user 20 μs, sys: 16 μs, total: 36 μs
Wall time: 37.9 μs
time: 551 μs (started: 2025-04-16 09:08:38 -07:00)


## Scrape web page content

In [ ]:
%%time
import requests
from bs4 import BeautifulSoup

def scrape_web_page(url):
    try:
        # Send HTTP request
        headers = {'User-Agent': 'Mozilla/5.0'}  # Avoid bot detection
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()  # Check for request errors

        # Parse HTML with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract text from paragraphs (customize as needed)
        paragraphs = soup.find_all('p')
        text = ' '.join([para.get_text().strip() for para in paragraphs])
        
        # Clean up: remove excessive whitespace
        text = ' '.join(text.split())
        #text = return text[:4000]  # Limit to ~4000 chars to avoid LLM token limits
        text = text[:2000]  # Limit to ~2000 chars to fit BERT's 512-token limit

        return text

    except requests.exceptions.SSLError as ssl_err:
        print(f"SSL Error: {ssl_err}. Try updating certifi or checking network settings.")
        return None
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

CPU times: user 6 μs, sys: 5 μs, total: 11 μs
Wall time: 11.9 μs
time: 1.31 ms (started: 2025-04-16 09:12:37 -07:00)


#### Test function

In [24]:
%%time
scrape_web_page("https://en.wikipedia.org/wiki/Artificial_intelligence")

CPU times: user 312 ms, sys: 24.8 ms, total: 337 ms
Wall time: 888 ms


'Artificial intelligence (AI) refers to the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1] Such machines may be called AIs. High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., ChatGPT and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general app

time: 889 ms (started: 2025-04-16 09:08:38 -07:00)


## Initialize LLM for question answering

In [25]:
%%time
from transformers import pipeline

def initialize_llm():
    # Use Hugging Face's question-answering pipeline
    # Model: distilbert-base-uncased-distilled-squad (lightweight, effective)
    try:
        llm_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")
        return llm_pipeline
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None

CPU times: user 7 μs, sys: 0 ns, total: 7 μs
Wall time: 8.11 μs
time: 563 μs (started: 2025-04-16 09:08:39 -07:00)


## Answer question based on web content

In [26]:
%%time

def answer_question_from_web(url, question):
    # Scrape content
    content = scrape_web_page(url)
    if not content:
        return "Failed to retrieve web content."

    # Initialize LLM
    llm_pipeline = initialize_llm()
    if not llm_pipeline:
        return "Failed to initialize LLM."

    # Use LLM to answer question
    try:
        result = llm_pipeline({
            'question': question,
            'context': content
        })
        answer = result['answer']
        score = result['score']
        return f"Answer: {answer} (Confidence: {score:.2%})"
    except Exception as e:
        return f"Error answering question: {e}"

CPU times: user 4 μs, sys: 9 μs, total: 13 μs
Wall time: 14.1 μs
time: 886 μs (started: 2025-04-16 09:08:39 -07:00)


## End-To-End testing

In [27]:
%%time

sample_url="https://finance.yahoo.com/news/nvidia-stock-dives-as-chipmaker-sees-55-billion-hit-from-surprise-china-chip-controls-130319576.html"
question="What is the reason for Nvidia's stock dive?"

answer_question_from_web(sample_url, question)

config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

Error initializing LLM: Failed to import transformers.models.distilbert.modeling_tf_distilbert because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.
CPU times: user 116 ms, sys: 52.1 ms, total: 168 ms
Wall time: 1.26 s


'Failed to initialize LLM.'

time: 1.27 s (started: 2025-04-16 09:08:39 -07:00)
